In [ ]:
print('===== NATIVE SOFTMAX CAUSAL FWD BITEXACT TEST =====')
print('===== NATIVE MERGEHBHEAD BITEXACT TEST =====')
# 1/4 - clone main + prototype (BiasAddRows .tkv)
import subprocess, os
print('===== NATIVE BIASADD BITEXACT TEST =====')
for d,br in [('/content/TVmain','main'),('/content/TVcpu','prototype/cuda-backend')]:
    subprocess.run('rm -rf '+d, shell=True)
    subprocess.run('git clone --depth 1 --branch '+br+' https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
    print(br, subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build native .so (has BiasAddRows) + .tkv-generated C .so
import subprocess, os
root='/content/TVmain/Release/tools/simd_native'
srcs='tkv_simd_native.c tkv_cuda_native.c tkv_linear_dw.c tkv_bwd_ops.c tkv_flash_attn.c tkv_step124.c tkv_prof.c tkv_attn_cuda.c'
p=subprocess.run('cd '+root+' && gcc -O2 -mavx2 -mfma -shared -fPIC '+srcs+' -lm -lpthread -ldl -o tkv_simd_native.so', shell=True, capture_output=True, text=True, timeout=600)
print('native build rc=',p.returncode)
ex='/content/TVcpu/Release/cuda_backend/examples'
cc=ex+'/merge_hb_head_cpu.cpu.c'; so=ex+'/merge_hb_head_cpu.so'
p2=subprocess.run('gcc -O2 -mavx2 -mfma -ffp-contract=off -fno-fast-math -shared -fPIC '+cc+' -o '+so+' -lm -lpthread -ldl', shell=True, capture_output=True, text=True, timeout=600)
print('tkv C build rc=',p2.returncode)
if p2.returncode!=0: print(p2.stderr[:2000])

In [ ]:
# 3/4 - bitwise compare SoftmaxCausalFwd
import ctypes, numpy as np
args=[ctypes.POINTER(ctypes.c_float),ctypes.c_int,ctypes.c_int,ctypes.c_double]
native=ctypes.CDLL('/content/TVmain/Release/tools/simd_native/tkv_simd_native.so').SoftmaxCausalFwd
native.argtypes=args; native.restype=ctypes.c_int
tkv=ctypes.CDLL('/content/TVcpu/Release/cuda_backend/examples/softmax_cf_cpu.so').SoftmaxCausalFwd
tkv.argtypes=args; tkv.restype=ctypes.c_int
rng=np.random.default_rng(7)
for n in [8,16,64,127]:
    s0=rng.standard_normal(n*n).astype(np.float32)
    a=s0.copy(); c=s0.copy()
    native(a.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),a.size,n,1.0)
    tkv(c.ctypes.data_as(ctypes.POINTER(ctypes.c_float)),c.size,n,1.0)
    d=np.count_nonzero(a.view(np.uint32)!=c.view(np.uint32))
    print('n=%d bitwise=%s diff=%d/%d maxabs=%.3g'%(n,d==0,d,a.size,np.abs(a-c).max()))

In [ ]:
# 4/4 - summary
print('If all bitwise=True -> BiasAddRows can be replaced by .tkv-generated C.')